# Ayomi — Foto Fokus Anak (YOLO11n-seg)

Notebook **final**, dijalankan dari atas ke bawah di Google Colab (GPU T4). Notebook ini menggantikan semua potongan sel yang sebelumnya diberikan di chat.

**Fungsi fitur.** Saat Pengasuh mengunggah foto logbook, model menemukan siluet setiap orang di foto, dengan kelas `anak` atau `dewasa`. Hasil akhirnya:
- anak target tetap tajam;
- latar diburamkan;
- orang lain diburamkan dan di-*pixelate*;
- setiap Orang Tua hanya menerima versi foto yang memfokuskan anaknya sendiri.

Model **tidak mengenali identitas**: tanpa pengenalan wajah dan tanpa biometrik. Anak target dipilih Pengasuh dengan satu ketukan, atau dipilih otomatis bila di foto hanya ada satu anak. Hasil selalu ditinjau Pengasuh sebelum terbit.

```
Colab (training) ──► Hugging Face model repo (privat, model.onnx)
                       └─► Hugging Face Space (FastAPI: /segment, /render) ◄── Route Handler Ayomi (Vercel)
```

> Status: notebook ini disusun dan diperiksa kodenya, tetapi belum pernah dijalankan utuh dari awal sampai akhir. Bagian bertanda ⚠️ perlu dicek saat dijalankan.

## Data yang dipakai

| Data | Jumlah | Cara mendapatkan | Dipakai untuk |
|---|---|---|---|
| **Open Images V7**: foto publik dengan mask orang berlabel Boy/Girl (anak) dan Man/Woman (dewasa) | ±7.000 diunduh lalu disaring → **3.000 train, 400 val, 300 test** | Otomatis (Sel 5), tanpa akun | Tahap 1 (training utama) |
| **Gambar daycare sintetis**, buatan AI, termasuk 13 gambar dari repo Amisah | **±150** | Dibuat tim dengan generator gambar | Tahap 2 (adaptasi ke suasana daycare) |
| **Foto ruangan kosong** (tanpa orang) | **10–20** | Difoto sendiri | Tahap 2 (contoh negatif) |
| **Foto nyata berizin** | **30–50** | Izin tertulis orang tua | **Hanya test**, tidak pernah dipakai untuk training |

Lisensi Open Images: anotasi CC BY 4.0, gambar CC BY 2.0. Cantumkan di dokumentasi. Basis model YOLO11n-seg (Ultralytics) berlisensi AGPL-3.0.

## Urutan kerja (3 sesi Colab)

Setiap sesi **selalu dimulai dengan Sel 1–3**. Sel yang pekerjaannya sudah selesai akan otomatis dilewati, karena hasilnya tersimpan di Drive.

| Sesi | Jalankan | Hasil | Butuh foto tim? |
|---|---|---|---|
| **A** | Sel 1–8 | Open Images tersimpan di cache Drive, model tahap 1 (±3 jam) | Tidak |
| *(di luar Colab)* | — | Buat gambar sintetis, kumpulkan foto berizin, unggah ke Drive | — |
| **B** | Sel 1–3, 9–10 | Foto domain dirapikan dan label awal untuk CVAT. Lalu **koreksi label di CVAT** | Ya |
| **C** | Sel 1–3, 5, 7, 11–20 | Tahap 2, evaluasi, ONNX, model dan API di Hugging Face | Ya |

Jika belum punya foto domain, Sesi C tetap bisa dijalankan. Evaluasi dan deploy akan memakai model tahap 1.

## Persiapan sekali saja: Hugging Face dan Colab Secrets

1. Daftar di huggingface.co, lalu buka **Settings → Access Tokens** dan buat dua token:
   - `ayomi-colab`, tipe **Write**, untuk upload model dan Space dari Colab;
   - `ayomi-space-read`, tipe **Fine-grained** dengan izin *read* untuk repo model saja. Token ini dipakai Space untuk membaca model privat.
2. Buat kunci API Ayomi, yaitu string acak minimal 32 karakter (misalnya dari password manager). Kunci yang sama nanti dipakai di Space dan di Vercel.
3. Di Colab, klik ikon 🔑 **Secrets**, tambahkan tiga secret berikut, lalu aktifkan *Notebook access* untuk ketiganya:

| Nama secret | Isi |
|---|---|
| `HF_TOKEN` | token `ayomi-colab` |
| `HF_READ_TOKEN` | token `ayomi-space-read` |
| `AYOMI_CV_TOKEN` | kunci API acak dari langkah 2 |

Jangan pernah menempel token di sel kode atau di chat.

**Runtime:** `Runtime → Change runtime type → T4 GPU`.

## Sel 1 — Instalasi

Colab sekarang memakai Python 3.13. Peringatan berikut **aman diabaikan**:
- `gradio ... requires pillow<13`: notebook ini tidak memakai Gradio;
- `SyntaxWarning: invalid escape sequence '\Z'`: berasal dari library `glob2`.

Jika Colab meminta *restart session* setelah instalasi, lakukan, lalu lanjut dari Sel 2. Jika nanti muncul error saat membaca gambar, jalankan `!pip -q install "pillow<13"` lalu restart session.

In [ ]:
!nvidia-smi -L
!pip -q install "ultralytics>=8.3" fiftyone onnx onnxslim onnxruntime huggingface_hub pyyaml

import sys, cv2, torch, PIL, ultralytics
print("Python", sys.version.split()[0], "| OpenCV", cv2.__version__, "| Pillow", PIL.__version__,
      "| Ultralytics", ultralytics.__version__, "| CUDA", torch.cuda.is_available())
assert torch.cuda.is_available(), "Aktifkan GPU: Runtime > Change runtime type > T4 GPU"

## Sel 2 — Konfigurasi

- `ROOT` (Google Drive) menyimpan hasil penting secara permanen: cache data, bobot model, metrik, dan foto domain.
- `WORK` (disk lokal Colab) lebih cepat, tetapi isinya hilang saat sesi berakhir.

**Perbaikan error `got multiple values for keyword argument 'dataset_dir'`:** FiftyOne versi baru tidak lagi menerima `dataset_dir` di `load_zoo_dataset()`. Folder unduhan sekarang diatur lewat `fo.config.dataset_zoo_dir` di sel ini.

Training dianggap selesai bila ada file penanda `SELESAI` di folder run-nya. File `best.pt` saja tidak cukup sebagai penanda, karena Ultralytics sudah menulis `best.pt` di tengah training. Dengan penanda ini, training yang terputus bisa dilanjutkan secara otomatis.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import hashlib, json, random, shutil, time, zipfile
from collections import Counter, defaultdict
from pathlib import Path
import cv2, numpy as np, pandas as pd, yaml
import matplotlib.pyplot as plt
from PIL import Image, ImageOps
import fiftyone as fo, fiftyone.zoo as foz
from ultralytics import YOLO

SEED = 42
random.seed(SEED); np.random.seed(SEED)
ROOT = Path("/content/drive/MyDrive/ayomi-focus")    # permanen (Drive)
WORK = Path("/content/work")                          # cepat, hilang saat sesi berakhir
for p in (ROOT, WORK, ROOT / "domain" / "raw", ROOT / "cache", ROOT / "runs"):
    p.mkdir(parents=True, exist_ok=True)

fo.config.dataset_zoo_dir = str(WORK / "fiftyone-zoo")   # pengganti parameter dataset_dir

CLASSES = ["anak", "dewasa"]   # urutan WAJIB sama dengan label di CVAT
OI_MAX = 7000                  # gambar Open Images berisi anak yang diunduh sebelum disaring
OI_ADULT_MAX = 5000            # gambar berisi orang dewasa (Man/Woman) yang diunduh sebelum disaring
N_OI_TRAIN, N_OI_VAL, N_OI_TEST, N_REPLAY = 3000, 400, 300, 400
N_ADULT_TRAIN, N_ADULT_VAL, N_ADULT_TEST = 1500, 150, 100   # tambahan gambar dewasa per split
IMGSZ = 640                    # resolusi input model
MAX_SIDE = 1600                # ukuran foto di aplikasi dan API
EVAL_SIDE = 1280               # resolusi saat menghitung metrik (lebih cepat)
LEAK_RATIO = 0.05              # >5% tubuh orang lain tetap tajam = bocor
AUTO_CONF = 0.5                # ambang pemilihan anak target otomatis
TARGET_FOUND = 0.95            # target: 95% anak dapat dijadikan fokus
DILATE_FRAC = 0.01             # pelebaran siluet target (1% sisi terpanjang)

STAGE1 = ROOT / "runs/stage1/weights/best.pt"
STAGE2 = ROOT / "runs/stage2/weights/best.pt"
done = lambda stage: (ROOT / "runs" / stage / "SELESAI").exists()
mark_done = lambda stage: (ROOT / "runs" / stage / "SELESAI").write_text(time.strftime("%Y-%m-%d %H:%M"))
print("Tahap 1 selesai:", done("stage1"), "| Tahap 2 selesai:", done("stage2"))

## Sel 3 — Fungsi bantu (gambar dan label YOLO)

Format label YOLO segmentasi: satu baris per orang, berisi `kelas x1 y1 x2 y2 …` dengan koordinat dinormalisasi 0–1. Sel ini berisi fungsi untuk membaca dan menulis label tersebut, serta untuk menampilkan siluet:
- hijau untuk anak;
- oranye untuk dewasa;
- biru muda untuk "orang" dari model COCO, yang belum membedakan umur.

In [ ]:
IMG_EXT = {".jpg", ".jpeg", ".png", ".webp"}
COLORS = {0: (80, 200, 120), 1: (0, 160, 255), -1: (255, 255, 0)}

def show(images, titles=None, size=6):
    plt.figure(figsize=(size * len(images), size))
    for i, img in enumerate(images):
        plt.subplot(1, len(images), i + 1); plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB)); plt.axis("off")
        if titles: plt.title(titles[i])
    plt.show()

def draw(img, persons):
    out = img.copy()
    for p in persons:
        cv2.polylines(out, [np.round(p["poly"]).astype(np.int32)], True, COLORS[p["cls"]], 3)
    return out

def read_seg_label(path, W, H):
    persons = []
    for line in (path.read_text().splitlines() if path and Path(path).exists() else []):
        v = line.split()
        if not v: continue
        if len(v) < 7 or len(v) % 2 == 0:
            raise ValueError(f"{Path(path).name}: bukan label poligon (export ulang sebagai segmentasi)")
        xy = np.asarray(v[1:], dtype=float).reshape(-1, 2) * [W, H]
        persons.append({"cls": int(v[0]), "poly": xy})
    return persons

def write_yolo_seg(recs, out_dir, split):
    for sub in ("images", "labels"):                     # bersihkan dulu agar tidak ada file sisa
        d = out_dir / sub / split
        shutil.rmtree(d, ignore_errors=True); d.mkdir(parents=True)
    for k, r in enumerate(recs):
        stem, src = f"{split}_{k:05d}", Path(r["file"])
        shutil.copy(src, out_dir / "images" / split / f"{stem}{src.suffix.lower()}")
        W, H = r["size"]
        lines = [f'{p["cls"]} ' + " ".join(f"{x / W:.5f} {y / H:.5f}" for x, y in p["poly"]) for p in r["persons"]]
        (out_dir / "labels" / split / f"{stem}.txt").write_text("\n".join(lines))

def load_yolo_split(root, split):
    recs = []
    for img in sorted((root / "images" / split).glob("*")):
        if img.suffix.lower() not in IMG_EXT: continue
        with Image.open(img) as im: W, H = im.size
        recs.append({"file": str(img), "size": (W, H),
                     "persons": read_seg_label(root / "labels" / split / f"{img.stem}.txt", W, H)})
    return recs

count = lambda recs, c: sum(p["cls"] == c for r in recs for p in r["persons"])

## Sel 4 — Uji akses Open Images (opsional, ±1 menit)

Sel ini mengunduh 20 foto saja, untuk memastikan akses berjalan dan mask anak tersedia. Tidak perlu akun, karena data diambil dari penyimpanan publik Google.

**Hasil yang diharapkan:** muncul baris seperti `segmentations: Boy (mask)` dan `segmentations: Girl (mask)` dengan angka lebih dari 0. Nama field bisa sedikit berbeda, tetapi harus ada field yang berlabel `(mask)`.

In [ ]:
probe = foz.load_zoo_dataset(
    "open-images-v7", split="validation", label_types=["detections", "segmentations"],
    classes=["Boy", "Girl"], max_samples=20, dataset_name="oi-probe", drop_existing_dataset=True)
print(probe)

counts = Counter()
for sample in probe:
    for name in sample.field_names:
        value = sample[name]
        if isinstance(value, fo.Detections):
            for d in value.detections:
                has_mask = getattr(d, "mask", None) is not None or getattr(d, "mask_path", None)
                counts[f"{name}: {d.label} ({'mask' if has_mask else 'kotak'})"] += 1
print(dict(sorted(counts.items())))

## Sel 5 — Open Images: unduh, saring, konversi, simpan cache

**Gambar yang dipakai** hanya yang memenuhi dua syarat:
- memuat anak (Boy/Girl);
- **semua** orang di dalamnya punya mask berlabel umur (Boy/Girl/Man/Woman).

Gambar dengan orang yang tidak punya mask dibuang. Orang yang tidak dilabeli akan mengajari model bahwa manusia adalah "latar".

**Label kembar:** orang yang sama bisa muncul di lebih dari satu field FiftyOne. Mask dengan kotak yang hampir sama (IoU ≥ 0,9) hanya disimpan sekali. Jika label kembarnya berbeda kelas (anak dan dewasa), gambar dibuang.

**Konversi:** mask Open Images disimpan per kotak (*crop*). Sel ini mengubahnya menjadi poligon dalam koordinat gambar penuh.

**Data dewasa:** gambar yang diunduh berdasarkan kelas Boy/Girl hampir semuanya berisi anak. Pada percobaan pertama, perbandingan anak dan dewasa ±33:1, sehingga kelas `dewasa` tidak terpelajari (mask AP50 0,04). Karena itu sel ini juga mengunduh gambar Man/Woman secara terpisah. Gambar dewasa tidak wajib memuat anak, tetapi semua orang di dalamnya tetap harus punya mask.

**Cache:** unduhan pertama memakan waktu ±45–90 menit. Hasil konversi lalu disimpan ke `ayomi-focus/cache/yolo_oi.zip` di Drive. Sesi berikutnya cukup memulihkan cache itu dalam ±2 menit. Jika `OI_MAX`, `OI_ADULT_MAX`, atau jumlah split diubah, hapus file cache tersebut supaya data dibuat ulang.

In [ ]:
HUMAN = {"Boy": 0, "Girl": 0, "Man": 1, "Woman": 1}

def det_mask(d):
    if getattr(d, "mask", None) is not None:
        return np.asarray(d.mask) > 0
    path = getattr(d, "mask_path", None)
    return cv2.imread(path, cv2.IMREAD_GRAYSCALE) > 0 if path else None

def box_iou(a, b):  # [x, y, w, h] relatif
    iw = max(0, min(a[0] + a[2], b[0] + b[2]) - max(a[0], b[0]))
    ih = max(0, min(a[1] + a[3], b[1] + b[3]) - max(a[1], b[1]))
    inter = iw * ih; union = a[2] * a[3] + b[2] * b[3] - inter
    return inter / union if union > 0 else 0.0

def mask_to_polygon(mask_crop, bbox, W, H):
    x0, y0 = int(round(bbox[0] * W)), int(round(bbox[1] * H))
    bw, bh = max(1, int(round(bbox[2] * W))), max(1, int(round(bbox[3] * H)))
    m = cv2.resize(mask_crop.astype(np.uint8), (bw, bh), interpolation=cv2.INTER_NEAREST)
    contours, _ = cv2.findContours(m, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not contours: return None
    c = max(contours, key=cv2.contourArea)
    if cv2.contourArea(c) < 25: return None
    c = cv2.approxPolyDP(c, 0.002 * cv2.arcLength(c, True), True).reshape(-1, 2).astype(float)
    if len(c) < 3: return None
    c[:, 0] = np.clip(c[:, 0] + x0, 0, W - 1); c[:, 1] = np.clip(c[:, 1] + y0, 0, H - 1)
    return c

def convert_sample(sample, require_child=True):
    with Image.open(sample.filepath) as im:
        W, H = im.size
    persons, unmatched = [], []
    for name in sample.field_names:
        value = sample[name]
        if not isinstance(value, fo.Detections): continue
        for d in value.detections:
            if d.label not in HUMAN and d.label != "Person": continue
            m = det_mask(d)
            if m is not None and d.label in HUMAN:
                poly = mask_to_polygon(m, d.bounding_box, W, H)
                if poly is not None:
                    # Orang yang sama bisa muncul di lebih dari satu field (label kembar): simpan sekali saja.
                    twin = next((p for p in persons if box_iou(p["bbox"], d.bounding_box) >= 0.9), None)
                    if twin is None:
                        persons.append({"cls": HUMAN[d.label], "poly": poly, "bbox": d.bounding_box})
                    elif twin["cls"] != HUMAN[d.label]:
                        return None      # satu orang berlabel anak sekaligus dewasa: label ambigu, buang gambar
                    continue
            unmatched.append(d.bounding_box)
    # Setiap kotak manusia harus berpasangan dengan mask berlabel umur; jika tidak, label tidak lengkap.
    if any(not any(box_iou(b, p["bbox"]) >= 0.7 for p in persons) for b in unmatched): return None
    if require_child and not any(p["cls"] == 0 for p in persons): return None
    if not persons: return None
    return {"file": sample.filepath, "size": (W, H), "persons": persons}

def download_oi(classes, max_samples, name):
    return foz.load_zoo_dataset(
        "open-images-v7", split="train", label_types=["detections", "segmentations"],
        classes=classes, only_matching=False, max_samples=max_samples, shuffle=True, seed=SEED,
        dataset_name=name, drop_existing_dataset=True)

def split_records(records, n_train, n_val, n_test):
    random.shuffle(records)
    n_test, n_val = min(n_test, len(records) // 10), min(n_val, len(records) // 10)
    return records[n_test + n_val:n_test + n_val + n_train], records[n_test:n_test + n_val], records[:n_test]

OI_DIR = WORK / "yolo_oi"
OI_CACHE = ROOT / "cache" / "yolo_oi.zip"
if (OI_DIR / "images" / "train").exists():
    print("Data Open Images sudah ada di sesi ini.")
elif OI_CACHE.exists():
    print("Memulihkan dari cache Drive ..."); shutil.unpack_archive(str(OI_CACHE), str(OI_DIR))
else:
    oi = download_oi(["Boy", "Girl"], OI_MAX, "ayomi-oi-children")
    print(oi)   # ⚠️ pastikan ada field Detections bermask (lihat Sel 4)
    kids = [r for r in (convert_sample(s) for s in oi.iter_samples(progress=True)) if r]
    need = N_OI_TRAIN + N_OI_VAL + N_OI_TEST
    print(f"Gambar anak lolos filter: {len(kids)} dari {len(oi)}")
    if len(kids) < need:
        print(f"⚠️ Kurang dari {need}. Naikkan OI_MAX (mis. 10000) di Sel 2 lalu jalankan ulang Sel 2 dan 5.")

    oi_adult = download_oi(["Man", "Woman"], OI_ADULT_MAX, "ayomi-oi-adults")
    seen = {r["file"] for r in kids}
    adults = [r for r in (convert_sample(s, require_child=False) for s in oi_adult.iter_samples(progress=True))
              if r and r["file"] not in seen and any(p["cls"] == 1 for p in r["persons"])]
    print(f"Gambar dewasa lolos filter: {len(adults)} dari {len(oi_adult)}")

    k_train, k_val, k_test = split_records(kids, N_OI_TRAIN, N_OI_VAL, N_OI_TEST)
    a_train, a_val, a_test = split_records(adults, N_ADULT_TRAIN, N_ADULT_VAL, N_ADULT_TEST)
    train, val, test = k_train + a_train, k_val + a_val, k_test + a_test
    for recs in (train, val, test): random.shuffle(recs)
    for recs, split in [(train, "train"), (val, "val"), (test, "test"), (train[:N_REPLAY], "replay")]:
        write_yolo_seg(recs, OI_DIR, split)
    shutil.make_archive(str(OI_CACHE.with_suffix("")), "zip", OI_DIR)
    print("Cache disimpan:", OI_CACHE)

oi_train, oi_val, oi_test = (load_yolo_split(OI_DIR, s) for s in ("train", "val", "test"))
OI_YAML = OI_DIR / "data.yaml"
OI_YAML.write_text(yaml.safe_dump({"path": str(OI_DIR), "train": "images/train", "val": "images/val",
                                   "test": "images/test", "names": dict(enumerate(CLASSES))}))
for name, recs in [("train", oi_train), ("val", oi_val), ("test", oi_test)]:
    print(f"{name}: {len(recs)} gambar | anak {count(recs, 0)} | dewasa {count(recs, 1)}")

## Sel 6 — Cek visual label

Siluet harus menempel rapi di tubuh. Siluet hijau untuk anak dan oranye untuk dewasa. Jika banyak siluet melenceng, berarti ada masalah konversi. Kirim gambarnya sebelum lanjut training.

In [ ]:
for r in random.sample(oi_train, 3):
    show([draw(cv2.imread(r["file"]), r["persons"])])

## Sel 7 — Fungsi prediksi, metrik, dan render

Metrik di sel ini meniru kejadian di aplikasi:

| Metrik | Arti | Target |
|---|---|---|
| `target_found` | % anak yang bisa dijadikan fokus (siluet prediksi cocok dengan aslinya, IoU ≥ 0,5) | ≥ 0,95 |
| `target_iou` | kerapian siluet anak | ≥ 0,85 |
| `leak_rate` | % foto berisi lebih dari satu orang yang masih menyisakan >5% tubuh orang lain dalam keadaan tajam | serendah mungkin |
| `class_acc` | ketepatan membedakan anak dan dewasa | setinggi mungkin |
| `auto_select` | pada foto berisi satu anak, % foto yang anak targetnya terpilih otomatis tanpa diketuk Pengasuh | setinggi mungkin |

`render_focus` sama persis dengan fungsi yang dipakai API (Sel 18), sehingga hasil di notebook identik dengan hasil di aplikasi. Jika salah satu diubah, ubah juga yang lain.

In [ ]:
def poly_mask(poly, W, H, scale=1.0):
    m = np.zeros((int(round(H * scale)), int(round(W * scale))), np.uint8)
    if poly is not None and len(poly) >= 3:
        cv2.fillPoly(m, [np.round(np.asarray(poly) * scale).astype(np.int32)], 1)
    return m.astype(bool)

def mask_iou(a, b):
    union = np.logical_or(a, b).sum()
    return np.logical_and(a, b).sum() / union if union else 0.0

def seg_predictor(weights, class_map=None):
    model = YOLO(str(weights), task="segment")
    def predict(img):
        r = model.predict(img, conf=0.05, iou=0.6, imgsz=IMGSZ, max_det=100, verbose=False)[0]
        if r.masks is None: return []
        out = []
        for poly, c, k in zip(r.masks.xy, r.boxes.conf.tolist(), r.boxes.cls.tolist()):
            k = int(k)
            if class_map is not None:
                if k not in class_map: continue
                k = class_map[k]
            if len(poly) >= 3: out.append({"cls": k, "conf": float(c), "poly": np.asarray(poly, dtype=float)})
        return out
    return predict

def predict_all(recs, fn):
    cache, times = {}, []
    for r in recs:
        img = cv2.imread(r["file"]); t = time.perf_counter()
        cache[r["file"]] = fn(img); times.append(time.perf_counter() - t)
    print(f"  {len(recs)} foto, {1000 * np.mean(times):.0f} ms/foto (GPU)")
    return cache

def focus_eval(recs, cache, conf, auto_conf=AUTO_CONF):
    t = defaultdict(float)
    for r in recs:
        W, H = r["size"]; sc = min(1.0, EVAL_SIDE / max(W, H))
        gts = [(p["cls"], poly_mask(p["poly"], W, H, sc)) for p in r["persons"]]
        preds = [p for p in cache[r["file"]] if p["conf"] >= conf]
        pms = [poly_mask(p["poly"], W, H, sc) for p in preds]
        kernel = np.ones((max(3, int(DILATE_FRAC * max(W, H) * sc)),) * 2, np.uint8)
        matched = set()
        for gc, gm in gts:                                   # ketepatan kelas dan orang palsu
            ious = [mask_iou(gm, pm) for pm in pms]
            if ious and max(ious) >= 0.5:
                j = int(np.argmax(ious)); matched.add(j)
                if preds[j]["cls"] >= 0:
                    t["cls_n"] += 1; t["cls_ok"] += preds[j]["cls"] == gc
        t["false_people"] += len(pms) - len(matched)
        children = [i for i, (gc, _) in enumerate(gts) if gc == 0]
        leak = False
        for gi in children:                                  # simulasi: setiap anak bergiliran menjadi target
            ious = [mask_iou(gts[gi][1], pm) for pm in pms]
            best = max(ious, default=0.0)
            t["targets"] += 1; t["iou_sum"] += best
            if best < 0.5: continue
            t["found"] += 1
            keep = cv2.dilate(pms[int(np.argmax(ious))].astype(np.uint8), kernel).astype(bool)
            for k, (_, om) in enumerate(gts):
                if k != gi and om.sum() and np.logical_and(om, keep).sum() / om.sum() > LEAK_RATIO:
                    leak = True
        if children and len(gts) > 1:
            t["photos_multi"] += 1; t["leak_photos"] += leak
        if len(children) == 1:
            t["single"] += 1
            cand = [j for j, p in enumerate(preds) if p["cls"] in (0, -1) and p["conf"] >= auto_conf]
            t["auto_ok"] += len(cand) == 1 and mask_iou(gts[children[0]][1], pms[cand[0]]) >= 0.5
    div = lambda a, b: round(t[a] / t[b], 4) if t[b] else None
    return {"target_found": div("found", "targets"), "target_iou": div("iou_sum", "targets"),
            "leak_rate": div("leak_photos", "photos_multi"), "class_acc": div("cls_ok", "cls_n"),
            "auto_select": div("auto_ok", "single"), "false_people": int(t["false_people"]),
            "targets": int(t["targets"]), "photos": len(recs)}

def pick_conf(recs, cache):
    df = pd.DataFrame([{"conf": float(c), **focus_eval(recs, cache, c)} for c in np.round(np.arange(0.1, 0.8, 0.05), 2)])
    ok = df[df.target_found >= TARGET_FOUND]
    best = (ok.sort_values(["leak_rate", "conf"], ascending=[True, False]) if len(ok)
            else df.sort_values("target_found", ascending=False)).iloc[0]
    if not len(ok): print("⚠️ target_found 95% belum tercapai")
    return float(best.conf), df

RENDER = {"blur_frac": 0.10, "pixel_block": 32, "dilate_frac": DILATE_FRAC, "feather_frac": 0.006}

def render_focus(bgr, target_mask, other_masks, extra_boxes=(), cfg=RENDER):
    H, W = bgr.shape[:2]
    k = max(31, int(min(H, W) * cfg["blur_frac"]) | 1)
    background = cv2.GaussianBlur(bgr, (k, k), 0)                          # latar: blur kuat
    kernel = np.ones((max(3, int(max(H, W) * cfg["dilate_frac"])),) * 2, np.uint8)
    keep = cv2.dilate(target_mask.astype(np.uint8), kernel)
    others = np.zeros((H, W), np.uint8)
    for m in other_masks: others |= m.astype(np.uint8)
    others = cv2.dilate(others, kernel).astype(bool)
    if others.any():                                                     # orang lain: blur + pixelate
        b = cfg["pixel_block"]
        small = cv2.resize(background, (max(1, W // b), max(1, H // b)), interpolation=cv2.INTER_AREA)
        background[others] = cv2.resize(small, (W, H), interpolation=cv2.INTER_NEAREST)[others]
    for x, y, w, h in extra_boxes:                                       # area blur manual dari Pengasuh
        keep[max(0, int(y)):int(y + h), max(0, int(x)):int(x + w)] = 0
    f = max(3, int(max(H, W) * cfg["feather_frac"]) | 1)
    alpha = cv2.GaussianBlur(keep.astype(np.float32), (f, f), 0)[..., None]   # tepi halus
    out = (alpha * bgr + (1 - alpha) * background).astype(np.uint8)
    keep_b = keep.astype(bool)
    overlap = any(m.sum() and np.logical_and(m, keep_b).sum() / m.sum() > LEAK_RATIO for m in other_masks)
    return out, (["overlap_other_person"] if overlap else [])

## Sel 8 — Training tahap 1 (Open Images)

Model dimulai dari bobot YOLO11n-seg pre-trained COCO, lalu dilatih ulang dengan dua kelas (`anak` dan `dewasa`). Perkiraan waktu sekitar 2–3 jam di T4.

**Jika sesi terputus:** jalankan ulang Sel 1–3, 5, dan 8. Sel ini otomatis melanjutkan dari `last.pt`.

In [ ]:
if not done("stage1"):
    last = ROOT / "runs/stage1/weights/last.pt"
    if last.exists():
        YOLO(str(last)).train(resume=True)
    else:
        YOLO("yolo11n-seg.pt").train(data=str(OI_YAML), epochs=50, imgsz=IMGSZ, batch=16, seed=SEED,
            project=str(ROOT / "runs"), name="stage1", exist_ok=True, patience=15, workers=2)
    mark_done("stage1")

m = YOLO(str(STAGE1)).val(data=str(OI_YAML), split="test", imgsz=IMGSZ, verbose=False)
print(f"Tahap 1 — mask mAP50: {m.seg.map50:.3f} | box mAP50: {m.box.map50:.3f}")
print("mask AP50 per kelas:", dict(zip(CLASSES, np.round(m.seg.ap50, 3))))

# Bagian 2 — Data domain daycare

Bagian ini dikerjakan **di luar Colab** dan bisa disiapkan sambil tahap 1 berjalan.

## 2a. Gambar sintetis (±150)

Pakai generator gambar yang biasa dipakai tim, lalu catat nama alatnya untuk laporan. Yang dicari adalah **variasi**, bukan keindahan. Template prompt:

```
Foto realistis suasana daycare di Indonesia, [JUMLAH] anak usia [USIA] sedang [KEGIATAN],
[PENGASUH: "dengan satu pengasuh dewasa di belakang" / tanpa pengasuh],
sudut kamera [setinggi mata anak / dari atas / dari samping], pencahayaan [terang siang / redup sore / lampu ruangan],
[KONDISI SULIT: "sebagian anak membelakangi kamera" / "anak saling berdekatan dan bertumpuk" / "anak di kejauhan"].
Tanpa teks, tanpa logo.
```

| Variabel | Pilihan |
|---|---|
| Jumlah anak | 1 (±40%), 2–3, 4–6 |
| Usia | bayi 1 tahun, balita 2–3 tahun, anak 4–6 tahun |
| Kegiatan | makan, tidur siang, bermain balok, mewarnai, kedatangan, bermain di luar |
| Pengasuh dewasa | ada di ±40% gambar (penting agar model belajar membedakan anak dan dewasa) |
| Kondisi sulit | minimal 30 dari 150 gambar |

## 2b. Foto nyata berizin (30–50, hanya untuk test)

Minta dari daycare mitra atau keluarga anggota tim. Surat izin minimal memuat:
- tujuan: menguji AI privasi foto;
- foto tidak dipublikasikan dan tidak dibagikan ke pihak lain;
- foto disimpan di Drive privat tim;
- foto dihapus setelah proyek selesai;
- tanda tangan orang tua.

**Jangan mengambil foto anak dari internet atau media sosial**, dan jangan memakai dataset anak yang asal-usulnya tidak jelas.

## 2c. Susunan folder di Drive

Nama subfolder pertama menentukan kolom `source` secara otomatis. Gambar yang dibuat dari prompt yang sama sebaiknya dimasukkan ke satu subfolder, supaya dihitung sebagai satu "sesi" dan tidak terpecah ke train dan test.

```
MyDrive/ayomi-focus/domain/raw/
├── sintetis/
│   ├── makan-01/*.jpg          ← satu subfolder per prompt (disarankan)
│   ├── amisah/*.jpg            ← 13 gambar dari repo Amisah
│   └── *.jpg                   ← tanpa subfolder: tiap gambar dianggap sesi sendiri
├── berizin/
│   └── 2026-11-20-kelas-a/*.jpg  ← satu subfolder per hari/kegiatan
└── ruangan_kosong/*.jpg
```

Format yang diterima: JPG, PNG, atau WebP. Foto HEIC dari iPhone perlu diubah ke JPG lebih dulu.

## Sel 9 — Rapikan foto domain

Sel ini memproses setiap foto di `raw/` sebagai berikut:
1. memutar foto sesuai orientasinya;
2. mengecilkan ke 1600 px, sama dengan ukuran di aplikasi;
3. **menghapus EXIF**, termasuk lokasi GPS;
4. menyimpan ke `prepared/` dengan nama dari hash isi file, misalnya `d_3f9a1c02be.jpg`, sehingga nama foto tidak berubah walaupun foto baru ditambahkan;
5. membuat atau memperbarui `domain_meta.csv`.

Di `domain_meta.csv`, kolom `source` dan `session` terisi otomatis. Kolom `lighting` boleh dibiarkan kosong; jika kosong, Sel 11 memperkirakannya dari kecerahan foto. Isian yang Anda ubah sendiri (`session`, `lighting`, `split`) tetap dipertahankan saat sel ini dijalankan ulang.

In [ ]:
RAW, PREP = ROOT / "domain" / "raw", ROOT / "domain" / "prepared"
META = ROOT / "domain" / "domain_meta.csv"
PREP.mkdir(parents=True, exist_ok=True)
SOURCES = {"sintetis", "berizin", "ruangan_kosong"}
KEEP_COLS = ("session", "lighting", "split")

old = pd.read_csv(META, dtype=str).fillna("").set_index("file") if META.exists() else None
rows, seen, skipped = [], set(), []
for src in sorted(p for p in RAW.rglob("*") if p.suffix.lower() in IMG_EXT):
    rel = src.relative_to(RAW).parts
    source = rel[0] if len(rel) > 1 else ""
    if source not in SOURCES:
        skipped.append("/".join(rel)); continue
    name = "d_" + hashlib.sha1(src.read_bytes()).hexdigest()[:10]
    if name in seen: continue                                   # duplikat persis
    seen.add(name)
    out = PREP / f"{name}.jpg"
    if not out.exists():
        with Image.open(src) as im:
            im = ImageOps.exif_transpose(im).convert("RGB")
            im.thumbnail((MAX_SIDE, MAX_SIDE))
            im.save(out, "JPEG", quality=90)                    # disimpan tanpa EXIF
    row = {"file": name, "original": "/".join(rel), "source": source,
           "session": "/".join(rel[:-1]) if len(rel) > 2 else f"{source}/{name}", "lighting": "", "split": ""}
    if old is not None and name in old.index:
        row.update({k: old.at[name, k] for k in KEEP_COLS if k in old.columns and old.at[name, k]})
    rows.append(row)

meta = pd.DataFrame(rows, columns=["file", "original", "source", "session", "lighting", "split"])
meta.to_csv(META, index=False)
print(meta.groupby("source").agg(foto=("file", "size"), sesi=("session", "nunique")))
if skipped:
    print(f"⚠️ {len(skipped)} file dilewati karena tidak berada di subfolder sintetis/berizin/ruangan_kosong:", skipped[:5])

## Sel 10 — Label awal otomatis untuk CVAT

Model tahap 1 membuat siluet awal untuk setiap foto, sehingga tugas Anda di CVAT tinggal **mengoreksi** (±20–40 detik per foto). Jika tahap 1 belum selesai, sel ini memakai `yolo11x-seg.pt` (model COCO besar). Dengan model itu, semua orang dilabeli `anak`, jadi yang dewasa harus Anda ubah secara manual.

### Langkah di CVAT (app.cvat.ai)

**Jangan pakai Roboflow paket gratis.** Proyeknya bersifat publik, sehingga foto anak bisa dilihat orang lain. CVAT bersifat privat secara default.

1. Buat **Project** dengan dua label **berurutan**: `anak` (pertama), lalu `dewasa`.
2. Unduh folder `MyDrive/ayomi-focus/domain/prepared` dari Drive, lalu buat **Task** baru dan unggah semua fotonya.
3. Buka **Actions → Upload annotations**, pilih format **Ultralytics YOLO Segmentation 1.0**, lalu unggah `MyDrive/ayomi-focus/domain/prelabel_cvat.zip`.
   - ⚠️ Nama format bisa sedikit berbeda antar versi CVAT. Pilih format YOLO *segmentation*, bukan *detection*.
   - Jika impor gagal, export dulu task kosong dengan format yang sama, lalu cocokkan struktur zip-nya dengan isi `prelabel_cvat.zip`.
4. **Koreksi semua foto.** Aturan pelabelan:
   - satu poligon per orang, mencakup seluruh tubuh yang terlihat, termasuk rambut dan pakaian;
   - **semua orang wajib dilabeli**, termasuk yang terpotong di tepi foto atau tampak dari belakang;
   - foto ruangan kosong dibiarkan tanpa label;
   - model sering melewatkan anak yang tertutup sebagian atau berada jauh di belakang. Label yang salah akan diajarkan ulang ke model, jadi periksa satu per satu.
   - *AI Tools → Segment Anything* (jika tersedia) membantu menambah orang yang terlewat.
5. Export dengan format yang sama dan centang **Save images**. Simpan hasilnya sebagai `MyDrive/ayomi-focus/domain/ayomi-domain-seg.zip`.

Setelah itu lanjutkan ke **Sesi C**: Sel 1–3, 5, 7, lalu 11–20.

In [ ]:
use_stage1 = done("stage1")
pre = YOLO(str(STAGE1) if use_stage1 else "yolo11x-seg.pt")
print("Label awal dari:", "model tahap 1" if use_stage1 else "COCO yolo11x-seg (semua orang = anak)")

OUT = WORK / "prelabel"
shutil.rmtree(OUT, ignore_errors=True)
for sub in ("images/train", "labels/train"): (OUT / sub).mkdir(parents=True)
meta = pd.read_csv(META, dtype=str).fillna("")
n_people = 0
for row in meta.to_dict("records"):
    img_path = PREP / f"{row['file']}.jpg"
    shutil.copy(img_path, OUT / "images/train" / img_path.name)
    lines = []
    if row["source"] != "ruangan_kosong":
        r = pre.predict(str(img_path), conf=0.25, imgsz=IMGSZ, verbose=False)[0]
        H, W = r.orig_shape
        if r.masks is not None:
            for poly, cls in zip(r.masks.xy, r.boxes.cls.tolist()):
                cls = int(cls)
                if not use_stage1:
                    if cls != 0: continue        # COCO: hanya kelas person
                    cls = 0                      # sementara dianggap anak; ubah yang dewasa di CVAT
                if len(poly) < 3: continue
                p = np.asarray(poly, np.float32).reshape(-1, 1, 2)
                p = cv2.approxPolyDP(p, 0.004 * cv2.arcLength(p, True), True).reshape(-1, 2)   # titik lebih sedikit = mudah dikoreksi
                if len(p) >= 3:
                    lines.append(f"{cls} " + " ".join(f"{x / W:.5f} {y / H:.5f}" for x, y in p))
    n_people += len(lines)
    (OUT / "labels/train" / f"{img_path.stem}.txt").write_text("\n".join(lines))

(OUT / "data.yaml").write_text(yaml.safe_dump({"path": ".", "train": "images/train", "names": dict(enumerate(CLASSES))}))
shutil.make_archive(str(ROOT / "domain" / "prelabel_cvat"), "zip", OUT)
print(f"{len(meta)} foto, {n_people} siluet awal → {ROOT / 'domain' / 'prelabel_cvat.zip'}")

# Bagian 3 — Tahap 2, evaluasi, dan deploy

## Sel 11 — Bagi split dan muat label hasil CVAT

**Aturan pembagian:**
- Semua foto `berizin` masuk **test**, tidak pernah dipakai untuk training.
- Gambar `sintetis` dan `ruangan_kosong` dibagi **per sesi** menjadi ±65% train, 20% valid, dan 15% test. Foto dari sesi yang sama hampir mirip; jika satu sesi terpecah ke train dan test, hasil tes menjadi terlalu bagus dan tidak jujur.
- Split yang sudah ditetapkan disimpan di `domain_meta.csv` dan tidak berubah saat foto baru ditambahkan.

**Foto yang dilewati:** foto tanpa file label di zip export dilewati dan dicetak namanya, kecuali foto ruangan kosong. Foto berisi orang tanpa label akan mengajari model bahwa orang adalah latar.

**Kolom otomatis:**
- `crowd` (`kosong`/`tunggal`/`kelompok`) dihitung dari label;
- `lighting`, jika kosong, diperkirakan dari kecerahan: rata-rata V < 90 berarti `redup`.

In [ ]:
DOMAIN_ZIP = ROOT / "domain" / "ayomi-domain-seg.zip"
DOM_DIR = WORK / "yolo_domain"
META = ROOT / "domain" / "domain_meta.csv"
PREP = ROOT / "domain" / "prepared"
HAS_DOMAIN = DOMAIN_ZIP.exists() and META.exists()
domain = {"train": [], "valid": [], "test": []}

if HAS_DOMAIN:
    meta = pd.read_csv(META, dtype=str).fillna("")
    # 1) Pembagian split
    real = meta.source == "berizin"
    meta.loc[real, "split"] = "test"
    target = {"train": 0.65, "valid": 0.20, "test": 0.15}
    pool = meta[~real]
    filled = Counter(pool.split[pool.split != ""])
    new = pool[pool.split == ""].groupby("session").size().sample(frac=1, random_state=SEED)
    for session, n in new.items():
        fixed = pool[(pool.session == session) & (pool.split != "")].split
        split = fixed.iloc[0] if len(fixed) else min(target, key=lambda s: filled[s] / max(1, len(pool)) - target[s])
        meta.loc[(meta.session == session) & ~real & (meta.split == ""), "split"] = split
        filled[split] += n
    meta.to_csv(META, index=False)

    # 2) Label dari export CVAT
    export = WORK / "domain_export"
    shutil.rmtree(export, ignore_errors=True)
    with zipfile.ZipFile(DOMAIN_ZIP) as z: z.extractall(export)
    for yml in export.rglob("*.yaml"):
        names = (yaml.safe_load(yml.read_text()) or {}).get("names")
        if names:
            names = [names[k] for k in sorted(names)] if isinstance(names, dict) else names
            assert [str(n).lower() for n in names[:2]] == CLASSES, f"Urutan kelas harus {CLASSES}, ditemukan {names}"
            break
    labels = {p.stem: p for p in export.rglob("*.txt") if "labels" in p.parts}
    missing = []
    for row in meta.to_dict("records"):
        img = PREP / f"{row['file']}.jpg"
        lab = labels.get(row["file"])
        if not img.exists() or (lab is None and row["source"] != "ruangan_kosong"):
            missing.append(row["file"]); continue
        with Image.open(img) as im: W, H = im.size
        persons = read_seg_label(lab, W, H)
        if not row["lighting"]:
            v = cv2.cvtColor(cv2.imread(str(img)), cv2.COLOR_BGR2HSV)[..., 2].mean()
            row["lighting"] = "redup" if v < 90 else "terang"
        row["crowd"] = "kosong" if not persons else "tunggal" if len(persons) == 1 else "kelompok"
        domain[row["split"]].append({"file": str(img), "size": (W, H), "meta": row, "persons": persons})
    if missing:
        print(f"⚠️ {len(missing)} foto tanpa label di export CVAT, dilewati:", missing[:10])
    if not domain["train"] or not domain["valid"]:
        print("⚠️ Train atau valid domain kosong; tahap 2 dilewati."); HAS_DOMAIN = False
    else:
        for split, recs in domain.items(): write_yolo_seg(recs, DOM_DIR, split)
        for split, recs in domain.items():
            by_source = dict(Counter(r["meta"]["source"] for r in recs))
            print(f"{split}: {len(recs)} foto {by_source} | anak {count(recs, 0)} | dewasa {count(recs, 1)}")
        for r in random.sample(domain["train"], min(2, len(domain["train"]))):
            show([draw(cv2.imread(r["file"]), r["persons"])])
else:
    print("Data domain belum ada; evaluasi dan deploy memakai model tahap 1.")

## Sel 12 — Training tahap 2 (adaptasi ke daycare)

Pengaturan fine-tune untuk data kecil:
- **Replay:** 400 gambar Open Images ikut dicampur agar model tidak "lupa" kemampuan umumnya.
- **`freeze=10`:** 10 lapisan awal dikunci.
- **Learning rate kecil:** `optimizer="SGD"` ditulis eksplisit, karena mode `auto` mengabaikan `lr0`.
- **Augmentasi untuk data kecil:**
  - `copy_paste` menempel siluet orang ke foto lain, sehingga satu anak muncul di banyak latar. Ini paling membantu untuk segmentasi.
  - `mixup` mencampur dua gambar.
  - Rotasi dan skala ditambah sedikit.

Jika data domain berubah setelah tahap 2 selesai, hapus folder `ayomi-focus/runs/stage2` agar training diulang.

In [ ]:
if HAS_DOMAIN and not done("stage2"):
    stage2_yaml = WORK / "stage2.yaml"
    stage2_yaml.write_text(yaml.safe_dump({
        "train": [str(DOM_DIR / "images/train"), str(OI_DIR / "images/replay")],
        "val": str(DOM_DIR / "images/valid"), "test": str(DOM_DIR / "images/test"),
        "names": dict(enumerate(CLASSES))}))
    last = ROOT / "runs/stage2/weights/last.pt"
    if last.exists():
        YOLO(str(last)).train(resume=True)
    else:
        YOLO(str(STAGE1)).train(
            data=str(stage2_yaml), epochs=60, imgsz=IMGSZ, batch=8, optimizer="SGD", lr0=0.002, freeze=10,
            copy_paste=0.3, mixup=0.1, degrees=5, scale=0.5, translate=0.1, fliplr=0.5,
            seed=SEED, project=str(ROOT / "runs"), name="stage2", exist_ok=True, patience=15, workers=2)
    mark_done("stage2")

BEST = STAGE2 if HAS_DOMAIN and done("stage2") else STAGE1
print("Model terbaik:", BEST)

## Sel 13 — Kalibrasi threshold dan evaluasi akhir

**Pembagian data evaluasi:**
- Threshold dipilih di set **valid**.
- Angka final diambil dari set **test**, yang tidak pernah dipakai untuk training maupun kalibrasi.
- Jika ada data domain, test berisi foto berizin ditambah sebagian gambar sintetis. Jika tidak ada, test memakai Open Images.

**Pembanding (`baseline_coco`):** YOLO11n-seg COCO tanpa training. COCO hanya mengenal "person", jadi `class_acc`-nya kosong. Angka baseline ini menjadi bukti di laporan bahwa training memang menambah kemampuan.

In [ ]:
EVAL_SET = domain["test"] if HAS_DOMAIN else oi_test
CALIB_SET = domain["valid"] if HAS_DOMAIN else oi_val
print(f"Data evaluasi: {'domain' if HAS_DOMAIN else 'Open Images'} | kalibrasi {len(CALIB_SET)} | test {len(EVAL_SET)}")

baseline = seg_predictor("yolo11n-seg.pt", class_map={0: -1})    # COCO person → kelas "tidak diketahui"
print("Baseline COCO — kalibrasi:"); BASE_CONF, df_base = pick_conf(CALIB_SET, predict_all(CALIB_SET, baseline))
best_fn = seg_predictor(BEST)
print("Model terlatih — kalibrasi:"); CONF, df_best = pick_conf(CALIB_SET, predict_all(CALIB_SET, best_fn))
display(df_best); print("threshold terpilih:", CONF)

models = {"baseline_coco": (baseline, BASE_CONF), "stage1": (seg_predictor(STAGE1), CONF)}
if BEST == STAGE2: models["stage2"] = (best_fn, CONF)
BEST_NAME = list(models)[-1]
caches, rows = {}, []
for name, (fn, conf) in models.items():
    print(name); caches[name] = predict_all(EVAL_SET, fn)
    rows.append({"model": name, "conf": conf, **focus_eval(EVAL_SET, caches[name], conf)})
results = pd.DataFrame(rows); display(results)

## Sel 14 — Analisis bias

Metrik dihitung terpisah per sumber foto, pencahayaan, dan keramaian. Ada dua hal yang **wajib dicatat** di dokumentasi (Technical Requirement):
- **Sumber:** jika model bagus di gambar `sintetis` tetapi turun di foto `berizin`, itu temuan penting dan tidak boleh disembunyikan.
- **Kondisi:** jika foto `redup` atau `kelompok` jauh lebih buruk, catat sebagai temuan bias.

Jumlah foto nyata masih kecil, jadi angka per kelompok punya ketidakpastian besar. Kolom `photos` menunjukkan jumlah fotonya.

In [ ]:
bias = {}
if HAS_DOMAIN:
    for key in ("source", "lighting", "crowd"):
        groups = defaultdict(list)
        for r in EVAL_SET: groups[r["meta"].get(key) or "-"].append(r)
        bias[key] = {g: focus_eval(recs, caches[BEST_NAME], CONF) for g, recs in groups.items()}
        print(f"\n== per {key} ==")
        display(pd.DataFrame(bias[key]).T)
else:
    print("Analisis bias membutuhkan data domain dengan kolom meta.")

## Sel 15 — Lihat hasil seperti yang akan diterima Orang Tua

Untuk setiap foto, anak dengan confidence tertinggi dijadikan target:
- gambar kiri menampilkan siluet hasil deteksi;
- gambar kanan menampilkan hasil fokus.

Perhatikan terutama foto kelompok dan foto yang diberi flag `overlap_other_person`. Di aplikasi, flag itu meminta Pengasuh memeriksa ulang hasilnya.

In [ ]:
for r in EVAL_SET[:6]:
    img = cv2.imread(r["file"]); H, W = img.shape[:2]
    preds = [p for p in best_fn(img) if p["conf"] >= CONF]
    kids = sorted([p for p in preds if p["cls"] == 0], key=lambda p: -p["conf"])
    if not kids:
        show([img], ["tidak ada anak terdeteksi → Pengasuh menandai manual / kirim tanpa foto"]); continue
    target = kids[0]
    others = [poly_mask(p["poly"], W, H) for p in preds if p is not target]
    out, flags = render_focus(img, poly_mask(target["poly"], W, H), others)
    show([draw(img, preds), out], [f"{len(preds)} orang terdeteksi", f"fokus anak ({target['conf']:.0%}) {flags}"])

## Sel 16 — Simpan metrik, export ONNX, cek kesetaraan

ONNX dijalankan di CPU Space tanpa perlu GPU. Untuk memastikan konversi tidak mengubah hasil, keluaran ONNX dibandingkan dengan model asli pada 30 foto. Selisih jumlah orang yang terdeteksi idealnya ≈ 0.

In [ ]:
MODEL_VERSION = "focus-yolo11n-seg-v1-" + time.strftime("%Y%m%d")
DOMAIN_COUNTS = {s: dict(Counter(r["meta"]["source"] for r in recs)) for s, recs in domain.items()}
metrics = {"model_version": MODEL_VERSION, "conf_threshold": CONF, "results": results.to_dict("records"), "bias": bias,
           "data": {"openimages": {"train": len(oi_train), "val": len(oi_val), "test": len(oi_test)},
                    "domain": DOMAIN_COUNTS}}
(ROOT / "metrics.json").write_text(json.dumps(metrics, indent=2, ensure_ascii=False, default=float))

onnx_path = YOLO(str(BEST)).export(format="onnx", imgsz=IMGSZ, opset=12, simplify=True)
onnx_fn = seg_predictor(onnx_path)
diffs = [abs(sum(p["conf"] >= CONF for p in best_fn(cv2.imread(r["file"])))
             - sum(p["conf"] >= CONF for p in onnx_fn(cv2.imread(r["file"])))) for r in EVAL_SET[:30]]
print("ONNX:", onnx_path, "| selisih jumlah orang rata-rata:", np.mean(diffs))

## Sel 17 — Upload model ke Hugging Face (repo privat)

Yang diunggah ke repo privat:
- `model.onnx`;
- `config.json`: threshold dan pengaturan render;
- `metrics.json`;
- model card.

Foto daycare **tidak** ikut diunggah.

In [ ]:
from google.colab import userdata
from huggingface_hub import HfApi, create_repo

TOKEN = userdata.get("HF_TOKEN")
api = HfApi(token=TOKEN); USER = api.whoami()["name"]
MODEL_REPO = f"{USER}/ayomi-focus-segmenter"
create_repo(MODEL_REPO, private=True, exist_ok=True, token=TOKEN)

n_syn = sum(c.get("sintetis", 0) for c in DOMAIN_COUNTS.values())
n_real = sum(c.get("berizin", 0) for c in DOMAIN_COUNTS.values())
config = {"model_version": MODEL_VERSION, "imgsz": IMGSZ, "conf_threshold": CONF, "auto_conf": AUTO_CONF,
          "classes": CLASSES, "max_side": MAX_SIDE, "leak_ratio": LEAK_RATIO, "render": RENDER}
card = f"""# Ayomi focus segmenter ({MODEL_VERSION})
Segmentasi orang (anak/dewasa) untuk fitur Foto Fokus Anak: anak target tajam, latar dan orang lain diburamkan.
Hasil WAJIB ditinjau Pengasuh. Model TIDAK mengenali identitas siapa pun.
- Basis: YOLO11n-seg (Ultralytics, AGPL-3.0).
- Data training: Open Images V7 (anotasi CC BY 4.0, gambar CC BY 2.0) {len(oi_train)} gambar; gambar daycare sintetis buatan AI {n_syn}.
- Data test: foto nyata berizin {n_real} (tidak dipublikasikan, tidak dipakai untuk training).
- Metrik dan bias: metrics.json. Batasan: anak bertumpuk dengan orang lain, cahaya redup, dan anak sangat kecil dapat gagal;
  jumlah foto nyata kecil sehingga angka evaluasi punya ketidakpastian besar.
"""
(WORK / "README.md").write_text(card); (WORK / "config.json").write_text(json.dumps(config, indent=2))
for local, remote in [(onnx_path, "model.onnx"), (WORK / "config.json", "config.json"),
                      (ROOT / "metrics.json", "metrics.json"), (WORK / "README.md", "README.md")]:
    api.upload_file(path_or_fileobj=str(local), path_in_repo=remote, repo_id=MODEL_REPO)
print("Model:", f"https://huggingface.co/{MODEL_REPO}")

## Sel 18 — Tulis file API (Hugging Face Space)

API ini punya dua endpoint utama:
- **`POST /segment`**: menerima foto, mengembalikan daftar orang (id, kelas, confidence, poligon). Jika hanya ada satu anak yang terdeteksi dengan yakin, `suggested_target_id` langsung terisi, sehingga Pengasuh tidak perlu mengetuk.
- **`POST /render`**: menerima foto, poligon anak target, dan area blur manual (opsional), lalu mengembalikan foto final dalam format JPEG base64.

**Pengamanan:**
- token dibandingkan dengan `hmac.compare_digest`;
- ukuran file dibatasi 5 MB;
- format dicek dari isi file, bukan dari nama;
- ada perlindungan dari *decompression bomb*;
- EXIF dibuang dari hasil;
- foto tidak disimpan di Space.

Versi Ultralytics di Space dikunci sama dengan versi di Colab, supaya hasilnya konsisten.

In [ ]:
import ultralytics
SPACE_DIR = WORK / "space"; SPACE_DIR.mkdir(exist_ok=True)

(SPACE_DIR / "app.py").write_text(r'''
import base64, hmac, io, json, os, time
import cv2
import numpy as np
from fastapi import FastAPI, File, Form, Header, HTTPException, UploadFile
from huggingface_hub import hf_hub_download
from PIL import Image, ImageOps, UnidentifiedImageError
from ultralytics import YOLO

Image.MAX_IMAGE_PIXELS = 40_000_000
MAX_BYTES = 5 * 1024 * 1024
FORMATS = {"JPEG", "PNG", "WEBP"}
REPO, API_TOKEN, READ = os.environ["MODEL_REPO"], os.environ["API_TOKEN"], os.environ.get("HF_READ_TOKEN")
CFG = json.load(open(hf_hub_download(REPO, "config.json", token=READ)))
MODEL = YOLO(hf_hub_download(REPO, "model.onnx", token=READ), task="segment")
app = FastAPI(title="Ayomi Foto Fokus Anak", version=CFG["model_version"])

def fail(status, code, message):
    raise HTTPException(status_code=status, detail={"code": code, "error": message})

def authorize(header):
    if not hmac.compare_digest(header or "", f"Bearer {API_TOKEN}"):
        fail(401, "unauthorized", "Token API tidak valid.")

async def read_image(file):
    data = await file.read(MAX_BYTES + 1)
    if not data: fail(400, "empty_file", "Berkas kosong.")
    if len(data) > MAX_BYTES: fail(413, "file_too_large", "Foto maksimal 5 MB.")
    try:
        with Image.open(io.BytesIO(data)) as im:
            fmt = im.format
            img = ImageOps.exif_transpose(im).convert("RGB")
    except (UnidentifiedImageError, Image.DecompressionBombError, OSError):
        fail(400, "invalid_image", "Berkas bukan gambar yang dapat dibaca.")
    if fmt not in FORMATS: fail(415, "unsupported_format", "Format harus JPG, PNG, atau WebP.")
    img.thumbnail((CFG["max_side"], CFG["max_side"]))
    return cv2.cvtColor(np.asarray(img), cv2.COLOR_RGB2BGR)

def poly_mask(poly, W, H):
    m = np.zeros((H, W), np.uint8)
    if len(poly) >= 3: cv2.fillPoly(m, [np.round(np.asarray(poly)).astype(np.int32)], 1)
    return m.astype(bool)

def mask_iou(a, b):
    union = np.logical_or(a, b).sum()
    return np.logical_and(a, b).sum() / union if union else 0.0

def segment(bgr):
    H, W = bgr.shape[:2]
    r = MODEL.predict(bgr, conf=CFG["conf_threshold"], iou=0.6, imgsz=CFG["imgsz"], max_det=50, verbose=False)[0]
    people = []
    if r.masks is not None:
        for poly, conf, cls in zip(r.masks.xy, r.boxes.conf.tolist(), r.boxes.cls.tolist()):
            if len(poly) < 3: continue
            p = np.asarray(poly, dtype=np.float32).reshape(-1, 1, 2)
            p = cv2.approxPolyDP(p, 0.003 * cv2.arcLength(p, True), True).reshape(-1, 2)
            if len(p) < 3: continue
            p[:, 0] = np.clip(p[:, 0], 0, W - 1); p[:, 1] = np.clip(p[:, 1], 0, H - 1)
            people.append({"class": CFG["classes"][int(cls)], "confidence": round(float(conf), 4),
                           "polygon": np.round(p).astype(int).tolist()})
    people.sort(key=lambda q: -q["confidence"])
    for i, q in enumerate(people): q["id"] = i
    return people

def render_focus(bgr, target_mask, other_masks, extra_boxes, cfg):
    H, W = bgr.shape[:2]
    k = max(31, int(min(H, W) * cfg["blur_frac"]) | 1)
    background = cv2.GaussianBlur(bgr, (k, k), 0)
    kernel = np.ones((max(3, int(max(H, W) * cfg["dilate_frac"])),) * 2, np.uint8)
    keep = cv2.dilate(target_mask.astype(np.uint8), kernel)
    others = np.zeros((H, W), np.uint8)
    for m in other_masks: others |= m.astype(np.uint8)
    others = cv2.dilate(others, kernel).astype(bool)
    if others.any():
        b = cfg["pixel_block"]
        small = cv2.resize(background, (max(1, W // b), max(1, H // b)), interpolation=cv2.INTER_AREA)
        background[others] = cv2.resize(small, (W, H), interpolation=cv2.INTER_NEAREST)[others]
    for x, y, w, h in extra_boxes:
        keep[max(0, int(y)):int(y + h), max(0, int(x)):int(x + w)] = 0
    f = max(3, int(max(H, W) * cfg["feather_frac"]) | 1)
    alpha = cv2.GaussianBlur(keep.astype(np.float32), (f, f), 0)[..., None]
    out = (alpha * bgr + (1 - alpha) * background).astype(np.uint8)
    keep_b = keep.astype(bool)
    overlap = any(m.sum() and np.logical_and(m, keep_b).sum() / m.sum() > CFG["leak_ratio"] for m in other_masks)
    return out, (["overlap_other_person"] if overlap else [])

def quality(bgr, people):
    H, W = bgr.shape[:2]
    flags = []
    if not people: flags.append("no_person_found")
    if any(q["confidence"] < CFG["auto_conf"] for q in people): flags.append("low_confidence")
    if sum(q["class"] == "anak" for q in people) > 1: flags.append("multiple_children")
    if min(W, H) < 320: flags.append("small_image")
    if cv2.Laplacian(cv2.cvtColor(bgr, cv2.COLOR_BGR2GRAY), cv2.CV_64F).var() < 60: flags.append("blurry")
    return flags

@app.get("/health")
def health():
    return {"status": "ok", "model_version": CFG["model_version"]}

@app.post("/segment")
async def segment_photo(file: UploadFile = File(...), authorization: str | None = Header(default=None)):
    authorize(authorization)
    bgr = await read_image(file); started = time.perf_counter()
    H, W = bgr.shape[:2]
    people = segment(bgr)
    kids = [q for q in people if q["class"] == "anak"]
    suggested = kids[0]["id"] if len(kids) == 1 and kids[0]["confidence"] >= CFG["auto_conf"] else None
    return {"image_size": {"width": W, "height": H}, "people": people, "suggested_target_id": suggested,
            "quality_flags": quality(bgr, people), "requires_human_review": True,
            "model_version": CFG["model_version"], "latency_ms": round((time.perf_counter() - started) * 1000)}

@app.post("/render")
async def render_photo(file: UploadFile = File(...), target_polygon: str = Form(...),
                       extra_blur_boxes: str = Form("[]"), authorization: str | None = Header(default=None)):
    authorize(authorization)
    bgr = await read_image(file); started = time.perf_counter()
    H, W = bgr.shape[:2]
    try:
        target = np.asarray(json.loads(target_polygon), dtype=float)
        boxes = json.loads(extra_blur_boxes)
        valid = (target.ndim == 2 and target.shape[1] == 2 and 3 <= len(target) <= 2000
                 and target[:, 0].min() >= 0 and target[:, 0].max() <= W
                 and target[:, 1].min() >= 0 and target[:, 1].max() <= H
                 and isinstance(boxes, list) and len(boxes) <= 20
                 and all(isinstance(b, list) and len(b) == 4 and all(isinstance(v, (int, float)) for v in b) for b in boxes))
    except Exception:
        valid = False
    if not valid:
        fail(400, "invalid_target", "target_polygon atau extra_blur_boxes tidak valid untuk ukuran foto ini.")
    target_mask = poly_mask(target, W, H)
    others = [m for m in (poly_mask(q["polygon"], W, H) for q in segment(bgr)) if mask_iou(m, target_mask) < 0.5]
    image, flags = render_focus(bgr, target_mask, others, boxes, CFG["render"])
    ok, jpeg = cv2.imencode(".jpg", image, [cv2.IMWRITE_JPEG_QUALITY, 88])   # tanpa EXIF
    return {"image_size": {"width": W, "height": H}, "other_people_blurred": len(others),
            "image": {"mime_type": "image/jpeg", "base64": base64.b64encode(jpeg.tobytes()).decode()},
            "quality_flags": ["manual_review_required", *flags], "requires_human_review": True,
            "model_version": CFG["model_version"], "latency_ms": round((time.perf_counter() - started) * 1000)}
''')

(SPACE_DIR / "requirements.txt").write_text(
    f"fastapi\nuvicorn[standard]\npython-multipart\nultralytics=={ultralytics.__version__}\nonnxruntime\nhuggingface_hub\npillow\n")
(SPACE_DIR / "Dockerfile").write_text('''FROM python:3.12-slim
RUN apt-get update && apt-get install -y --no-install-recommends libgl1 libglib2.0-0 && rm -rf /var/lib/apt/lists/*
RUN useradd -m -u 1000 user
USER user
ENV HOME=/home/user PATH=/home/user/.local/bin:$PATH
WORKDIR /home/user/app
COPY --chown=user requirements.txt .
RUN pip install --no-cache-dir --user torch torchvision --index-url https://download.pytorch.org/whl/cpu \\
 && pip install --no-cache-dir --user -r requirements.txt
COPY --chown=user app.py .
CMD ["uvicorn", "app:app", "--host", "0.0.0.0", "--port", "7860"]
''')
(SPACE_DIR / "README.md").write_text("---\ntitle: Ayomi Foto Fokus Anak\nsdk: docker\napp_port: 7860\n---\nAPI segmentasi & fokus anak untuk Ayomi.\n")
print(sorted(p.name for p in SPACE_DIR.iterdir()))

## Sel 19 — Deploy Space

Sel ini membuat Space bertipe Docker, lalu mengisi:
- variabel `MODEL_REPO`;
- secret `API_TOKEN` dan `HF_READ_TOKEN`, yang diambil dari Colab Secrets dan tidak pernah dicetak.

Build memakan waktu ±5–10 menit. Pantau prosesnya di tab **Logs** halaman Space.

In [ ]:
SPACE_REPO = f"{USER}/ayomi-foto-fokus"
create_repo(SPACE_REPO, repo_type="space", space_sdk="docker", exist_ok=True, token=TOKEN)
api.add_space_variable(SPACE_REPO, "MODEL_REPO", MODEL_REPO)
api.add_space_secret(SPACE_REPO, "API_TOKEN", userdata.get("AYOMI_CV_TOKEN"))
api.add_space_secret(SPACE_REPO, "HF_READ_TOKEN", userdata.get("HF_READ_TOKEN"))
api.upload_folder(folder_path=str(SPACE_DIR), repo_id=SPACE_REPO, repo_type="space")
print("Space:", f"https://huggingface.co/spaces/{SPACE_REPO}")

## Sel 20 — Uji API end-to-end

Jalankan setelah status Space menjadi **Running**.

Kode status yang diharapkan:
- `segment` → 200;
- tanpa token → 401;
- bukan gambar → 400;
- terlalu besar → 413.

⚠️ URL Space bisa berbeda dari tebakan kode. Cek di menu `⋮ → Embed this Space → Direct URL`, lalu sesuaikan `SPACE_URL` bila perlu.

In [ ]:
import base64, requests

SPACE_URL = f"https://{SPACE_REPO.replace('/', '-').replace('_', '-').replace('.', '-').lower()}.hf.space"
HEADERS = {"Authorization": f"Bearer {userdata.get('AYOMI_CV_TOKEN')}"}
sample_path = EVAL_SET[0]["file"]
print("health:", requests.get(f"{SPACE_URL}/health", timeout=180).json())

with open(sample_path, "rb") as f:
    seg = requests.post(f"{SPACE_URL}/segment", headers=HEADERS, files={"file": f}, timeout=120)
print("segment:", seg.status_code); seg = seg.json()
print({k: v for k, v in seg.items() if k != "people"}, "| orang:", [(p["id"], p["class"], p["confidence"]) for p in seg["people"]])

kids = [p for p in seg["people"] if p["class"] == "anak"]
target = next((p for p in seg["people"] if p["id"] == seg["suggested_target_id"]), kids[0] if kids else None)
if target:
    with open(sample_path, "rb") as f:
        ren = requests.post(f"{SPACE_URL}/render", headers=HEADERS, files={"file": f},
                            data={"target_polygon": json.dumps(target["polygon"])}, timeout=120).json()
    print({k: v for k, v in ren.items() if k != "image"})
    show([cv2.imdecode(np.frombuffer(base64.b64decode(ren["image"]["base64"]), np.uint8), cv2.IMREAD_COLOR)], ["hasil untuk Orang Tua"])

with open(sample_path, "rb") as f:
    print("tanpa token:", requests.post(f"{SPACE_URL}/segment", files={"file": f}, timeout=60).status_code)
print("bukan gambar:", requests.post(f"{SPACE_URL}/segment", headers=HEADERS, files={"file": ("x.jpg", b"bukan gambar")}, timeout=60).status_code)
print("terlalu besar:", requests.post(f"{SPACE_URL}/segment", headers=HEADERS, files={"file": ("b.jpg", b"0" * (6 * 1024 * 1024))}, timeout=60).status_code)

# Lampiran

## A. Hugging Face setelah deploy

| Hal | Yang perlu diketahui |
|---|---|
| **Melihat error** | Halaman Space → tab **Logs**. Ada dua jenis log: *Build* (instalasi) dan *Container* (saat API berjalan) |
| **Space tidur** | Space gratis tidur setelah ±48 jam tanpa permintaan. Permintaan pertama sesudahnya bisa memakan 1–2 menit. Untuk demo, buka `/health` beberapa menit sebelumnya |
| **Spesifikasi** | CPU basic gratis (2 vCPU, 16 GB RAM). Ukur kecepatan sungguhan dari `latency_ms` di Sel 20 |
| **Memperbarui model** | Jalankan ulang Sel 16–17, lalu buka **Settings → Factory rebuild** di halaman Space, atau jalankan ulang Sel 19 |
| **Ganti kunci API** | Perbarui secret `API_TOKEN` di **Settings → Variables and secrets** Space, lalu perbarui juga di Vercel |
| **Privasi** | Repo model bersifat privat. Space publik hanya memperlihatkan kode, dan setiap permintaan wajib membawa token. Foto hanya diproses lalu dikembalikan, tidak disimpan |

## B. Integrasi ke Ayomi (dikerjakan setelah API berjalan)

**1. Variabel server** di Vercel dan `.env.local`. Jangan pakai awalan `NEXT_PUBLIC_`.

```
CV_API_URL=https://<user>-ayomi-foto-fokus.hf.space
CV_API_TOKEN=<sama dengan AYOMI_CV_TOKEN>
```

**2. Alur di aplikasi:**

```
Browser Pengasuh
 1. Pilih foto → browser mengecilkan ke maks 1600 px JPEG (±300–700 KB; batas body Vercel 4,5 MB)
 2. Route Handler: cek Pengasuh ditugaskan → simpan asli ke bucket privat "logbook-originals"
    → CV /segment → daftar orang + suggested_target_id
 3. UI menggambar siluet di atas foto. Kasus satu anak: otomatis terpilih.
 4. Saat Publikasikan: untuk setiap anak yang dicatat → CV /render (poligon anak itu)
    → simpan ke bucket "logbook-masked" sebagai attachment_variants (audience_child_id)
 5. Orang Tua hanya melihat varian anaknya, lewat signed URL berumur pendek
```

**3. Jika proses gagal**, misalnya Space tidur atau timeout: tampilkan "AI sedang disiapkan, coba lagi", atau beri pilihan mengirim catatan tanpa foto. **Foto asli tidak pernah dipakai sebagai pengganti.**

**4. Kontrak AI:** `privacy_masking` perlu diperbarui dari kebijakan `all_faces` ke `focus_target`, dengan output berupa poligon dan satu varian per anak.

## C. Error yang sudah diketahui

| Error | Penyebab | Solusi |
|---|---|---|
| `got multiple values for keyword argument 'dataset_dir'` | FiftyOne baru tidak menerima `dataset_dir` | Sudah diperbaiki: `fo.config.dataset_zoo_dir` di Sel 2 |
| `gradio ... requires pillow<13` | Konflik dengan paket bawaan Colab | Abaikan. Jika gambar gagal dibaca: `pip install "pillow<13"` lalu restart |
| `Lolos filter` jauh di bawah 3.700 | Banyak gambar dengan orang tanpa mask | Naikkan `OI_MAX`, hapus `cache/yolo_oi.zip`, lalu jalankan ulang Sel 2 dan 5 |
| `Urutan kelas harus ['anak', 'dewasa']` | Urutan label di Project CVAT salah | Ubah urutan label di CVAT, lalu export ulang |
| `bukan label poligon` | Export memakai format *detection* | Export ulang dengan format YOLO *segmentation* |
| Sesi Colab terputus saat training | Batas waktu Colab | Jalankan Sel 1–3, 5, lalu sel training lagi; training otomatis dilanjutkan |